# 数字水印工具包：综合应用与完整流程 — 自学课程

**分类**：数字水印

**内容简介**：整合扩频水印核心功能，提供测试图像生成、水印图案创建、嵌入提取、可视化分析及报告生成的一站式工具包

> 提示：本课程代码尽量只使用 Python 标准库，以兼容 Pyodide 运行环境。



## 学习目标

完成学习后，你将能够：

- 用自己的话解释核心概念与安全直觉
- 写出可运行的最小实现（加密/解密/验证/攻击/评估）
- 通过小实验验证关键性质，并能定位常见错误
- 完成练习题：从“会用”走向“会分析”



## 使用方式与学习节奏（建议）

- 第 1 遍：快速通读文字 + 运行全部代码
- 第 2 遍：边运行边改参数，观察输出变化
- 第 3 遍：完成练习题；对照“调试提示”定位问题

> 学习时尽量把每个 Step 当作一个“小目标”，做完自检题再进入下一步。



# Step 1：把水印系统当成“管线（pipeline）”

一个完整水印系统可以拆成 6 个模块：

1) Watermark 生成：$w\leftarrow\mathrm{Gen}()$
2) Key/Seed 管理：$k$
3) Embedding：$x' = \mathrm{Embed}(x,w,k)$
4) 攻击通道：$\tilde{x}=\mathrm{Attack}(x')$
5) Extraction/Detection：$\hat{w}=\mathrm{Extract}(\tilde{x},k)$
6) 评估：PSNR、BER、检测率等

> 本课程用“字节数组载体”统一演示：LSB（空域）+ DCT（频域 toy）+ 评估函数。



In [ ]:
# Step 1：通用评估函数
def ber(bits, bits_hat):
    if len(bits)!=len(bits_hat):
        raise ValueError("长度不一致")
    wrong=sum(1 for a,b in zip(bits,bits_hat) if a!=b)
    return wrong/len(bits)

def confusion(tp, fp, tn, fn):
    acc = (tp+tn)/(tp+fp+tn+fn) if (tp+fp+tn+fn)>0 else 0
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec = tp/(tp+fn) if (tp+fn)>0 else 0
    return acc, prec, rec



# Step 2：空域 LSB 模块复用

我们复用前面 LSB 的 Embed/Extract，并加入一个“检测器”：当提取文本能正确解码并匹配预期前缀时，判定检测成功。



In [ ]:
# Step 2：LSB 模块（复用）
def bits_from_text(s: str):
    b = s.encode("utf-8")
    out=[]
    for byte in b:
        for i in range(7,-1,-1):
            out.append((byte>>i)&1)
    return out

def text_from_bits(bits):
    if len(bits)%8!=0:
        raise ValueError("bits 长度需为 8 的倍数")
    out=bytearray()
    for i in range(0,len(bits),8):
        byte=0
        for j in range(8):
            byte=(byte<<1)|bits[i+j]
        out.append(byte)
    return out.decode("utf-8", errors="replace")

def lsb_embed(carrier, bits):
    if len(bits)>len(carrier):
        raise ValueError("载体太短")
    out=carrier[:]
    for i,b in enumerate(bits):
        out[i]=(out[i]&~1)|int(b)
    return out

def lsb_extract(stego, n_bits):
    return [(stego[i]&1) for i in range(n_bits)]

def detect_prefix(stego, n_bits, prefix="WM:"):
    bits=lsb_extract(stego, n_bits)
    txt=text_from_bits(bits)
    return txt.startswith(prefix), txt



# Step 3：攻击通道库（可组合）

我们提供三个最小攻击：
- 噪声扰动
- 截断（裁剪）
- 量化（模拟压缩）

并统一成 `attack_pipeline`。



In [ ]:
# Step 3：攻击库
def add_noise(arr, p=0.05):
    out=arr[:]
    for i in range(len(out)):
        if random.random()<p:
            out[i]=max(0, min(255, out[i] + (1 if random.random()<0.5 else -1)))
    return out

def crop(arr, keep_ratio=0.9):
    n=int(len(arr)*keep_ratio)
    return arr[:n] + [0]*(len(arr)-n)

def quantize_bytes(arr, q=4):
    out=[]
    for x in arr:
        out.append(int(round(x/q)*q))
    return out

def attack_pipeline(arr, noise_p=0.0, crop_ratio=1.0, q=None):
    out=arr[:]
    if noise_p>0:
        out=add_noise(out, p=noise_p)
    if crop_ratio<1.0:
        out=crop(out, keep_ratio=crop_ratio)
    if q is not None:
        out=quantize_bytes(out, q=q)
    return out



# Step 4：端到端实验：嵌入 → 攻击 → 提取/检测 → 评估

我们用固定前缀 `WM:` 作为“是否检测到水印”的判定。



In [ ]:
# Step 4：端到端实验
carrier=[secrets.randbelow(256) for _ in range(500)]
wm_text="WM:HELLO"
bits=bits_from_text(wm_text)
stego=lsb_embed(carrier, bits)

# 攻击
att=attack_pipeline(stego, noise_p=0.15, crop_ratio=1.0, q=4)

ok, txt = detect_prefix(att, len(bits), prefix="WM:")
print("detected=", ok, "extracted=", txt)
print("PSNR=", round(psnr(carrier, stego), 2), "BER=", ber(bits, lsb_extract(att, len(bits))))
# 预期输出：detected 可能 True/False（受攻击强度影响），并输出 PSNR/BER



# Step 5：参数扫描（小实验）

我们扫描噪声概率 $p$，记录检测率与 BER 的变化。



In [ ]:
# Step 5：扫描噪声强度
def run_trials(p, trials=20):
    ok_cnt=0
    ber_list=[]
    for _ in range(trials):
        att=attack_pipeline(stego, noise_p=p, q=4)
        ok,_=detect_prefix(att, len(bits), prefix="WM:")
        ok_cnt += 1 if ok else 0
        ber_list.append(ber(bits, lsb_extract(att, len(bits))))
    return ok_cnt/trials, sum(ber_list)/len(ber_list)

for p in [0.0, 0.05, 0.1, 0.15, 0.2]:
    rate, b = run_trials(p, trials=30)
    print(p, "detect_rate=", round(rate,2), "avg_BER=", round(b,3))
# 预期输出：p 越大，detect_rate 往往下降，BER 上升



## 练习

1) 把检测器从“前缀匹配”改成“校验和匹配”（例如在水印里带一个 short hash）。
2) 引入重复编码（每比特 3 次投票），观察 detect_rate 是否提升。
3) 思考：为什么真实水印系统会使用纠错码 + 同步码（sync）？



## 总结与延伸

- 你已经完成了核心概念、最小实现与小实验。
- 建议继续：
  - 把实现改成支持更多字符集/更大规模输入
  - 给代码加上单元测试（你也可以用 assert 作为轻量测试）
  - 思考：如果攻击者更强/网络更复杂/随机数更弱，会发生什么？

> 进阶的关键不是“代码更长”，而是“能解释每一步为什么安全/为什么不安全”。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”

